In [1]:
import os
import numpy as np
import pandas as pd
import faiss
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("API key terbaca:", os.getenv("GEMINI_API_KEY") is not None)

API key terbaca: True


In [2]:
df = pd.read_csv("../data/ecommerce_faq_knowledge_base.csv")
print("Jumlah entri:", len(df))
print("Kolom:", df.columns.tolist())
df.head()

Jumlah entri: 59
Kolom: ['faq_id', 'category', 'question', 'answer']


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [3]:
print(df.isna().sum())
print()
print(df.iloc[0].to_dict())

faq_id      0
category    0
question    0
answer      0
dtype: int64

{'faq_id': 1, 'category': 'Pemesanan', 'question': 'Bagaimana cara melakukan pemesanan di platform ini?', 'answer': "Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran."}


In [4]:
print(df["category"].value_counts())

category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64


In [5]:
def build_text(row):
    return (
         f"Kategori: {row['category']}\n"
        f"Pertanyaan: {row['question']}\n"
        f"Jawaban: {row['answer']}"
    )

df["doc_text"] = df.apply(build_text, axis=1)
print(df["doc_text"].iloc[0])
    

Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


In [6]:
EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-2.5-flash"
EMBED_DIM = 768

In [7]:
def embed(texts, task_type, batch_size=50):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        res = client.models.embed_content(
            model=EMBED_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=EMBED_DIM,
            ),
        )
        all_vecs.extend(e.values for e in res.embeddings)
    vecs = np.array(all_vecs, dtype="float32")
    faiss.normalize_L2(vecs)
    return vecs

In [8]:
doc_vecs = embed(df["doc_text"].tolist(), "RETRIEVAL_DOCUMENT")
print("Shape embedding:", doc_vecs.shape)
assert doc_vecs.shape == (len(df), EMBED_DIM)
print("OK, shape sesuai")

Shape embedding: (59, 768)
OK, shape sesuai


In [9]:
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(doc_vecs)

print("Jumlah vektor di index:", index.ntotal)
print("Jumlah entri knowledge base:", len(df))
assert index.ntotal == len(df)
print("OK, jumlah sesuai")

Jumlah vektor di index: 59
Jumlah entri knowledge base: 59
OK, jumlah sesuai


In [10]:
faiss.write_index(index, "../data/faq.index")
print("Index tersimpan")

Index tersimpan


In [11]:
def semantic_search(query, k=3):
    q_vec = embed([query], "RETRIEVAL_QUERY")
    scores, ids = index.search(q_vec, k)

    results = []
    for score, i in zip(scores[0], ids[0]):
        row = df.iloc[i]
        results.append({
            "faq_id": int(row["faq_id"]),
            "score": float(score),
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
        })
    return results

In [12]:
def tampilkan(query, k=3):
    print("QUERY:", query)
    for r in semantic_search(query, k):
        print(f"  {r['score']:.3f} | [{r['category']}] {r['question']}")
    print()

tampilkan("Paket saya kok belum nyampe-nyampe?")
tampilkan("Kalau barangnya rusak, uangnya bisa balik nggak?")
tampilkan("Apakah kalian menjual tiket konser?")
tampilkan("Bagaimana cara membatalkan pesanan yang sudah dibuat?")
tampilkan("Apakah saya bisa mengubah alamat pengiriman sesuai dengan yang saya inginkan?")

QUERY: Paket saya kok belum nyampe-nyampe?
  0.777 | [Pengiriman] Bagaimana jika paket saya belum sampai melebihi estimasi?
  0.765 | [Pengiriman] Kenapa status pengiriman saya tidak berubah selama beberapa hari?
  0.734 | [Pengiriman] Kenapa nomor resi saya belum bisa dilacak di website kurir?

QUERY: Kalau barangnya rusak, uangnya bisa balik nggak?
  0.738 | [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
  0.732 | [Pengembalian] Apakah biaya ongkos kirim pengembalian ditanggung pembeli?
  0.723 | [Produk] Bagaimana jika produk yang saya pesan ternyata kehabisan stok?

QUERY: Apakah kalian menjual tiket konser?
  0.620 | [Pemesanan] Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller?
  0.606 | [Pemesanan] Bagaimana cara melakukan pemesanan di platform ini?
  0.605 | [Produk] Apakah stok yang tertera di aplikasi selalu update secara real-time?

QUERY: Bagaimana cara membatalkan pesanan yang sudah dibuat?
  0.782 | [Pemesanan] Bagaimana cara membat

In [13]:
def build_prompt(query, results):
    context = "\n\n".join(
        f"[FAQ {n+1}] Kategori: {r['category']}\n"
        f"Pertanyaan: {r['question']}\n"
        f"Jawaban: {r['answer']}"
        for n, r in enumerate(results)
    )

    return f"""# Persona
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan ringkas.

# Task
Jawab pertanyaan pelanggan HANYA berdasarkan bagian Context di bawah.
Jangan memakai pengetahuan umum dan jangan mengarang informasi.
Jika Context tidak memuat jawaban atas pertanyaan, katakan dengan jujur bahwa
informasi tersebut tidak tersedia di FAQ, lalu sarankan pelanggan menghubungi tim customer service.

# Context
{context}

# Format
- Gunakan Bahasa Indonesia dengan nada ramah dan profesional.
- Maksimal 3-4 kalimat. Gunakan poin jika ada langkah-langkah.
- Jangan menyebut kata "Context" atau nomor FAQ dalam jawaban.

# Pertanyaan Pelanggan
{query}

# Jawaban"""

In [14]:
hasil = semantic_search("Paket saya kok belum nyampe-nyampe?", k=3)
print(build_prompt("Paket saya kok belum nyampe-nyampe?", hasil))

# Persona
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan ringkas.

# Task
Jawab pertanyaan pelanggan HANYA berdasarkan bagian Context di bawah.
Jangan memakai pengetahuan umum dan jangan mengarang informasi.
Jika Context tidak memuat jawaban atas pertanyaan, katakan dengan jujur bahwa
informasi tersebut tidak tersedia di FAQ, lalu sarankan pelanggan menghubungi tim customer service.

# Context
[FAQ 1] Kategori: Pengiriman
Pertanyaan: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

[FAQ 2] Kategori: Pengiriman
Pertanyaan: Kenapa status pengiriman saya tidak berubah selama beberapa hari?
Jawaban: Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.

[FAQ 3] Kategori: Pengiriman
Pertanyaan: Kenapa nomor 

In [15]:
def answer_query(query, k=3, verbose=True):
    # 1. Retrieval
    results = semantic_search(query, k)

    # 2. Prompt
    prompt = build_prompt(query, results)

    # 3. Generate
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    )

    if verbose:
        print("QUERY:", query)
        print("FAQ yang ditemukan:")
        for r in results:
            print(f"  {r['score']:.3f} | {r['question']}")
        print("\nJAWABAN AI:")
        print(response.text)
        print("-" * 60)

    return results, response.text

In [16]:
_ = answer_query("Paket saya dimana, ko nda nyampe?")

QUERY: Paket saya dimana, ko nda nyampe?
FAQ yang ditemukan:
  0.758 | Bagaimana jika paket saya belum sampai melebihi estimasi?
  0.737 | Kenapa status pengiriman saya tidak berubah selama beberapa hari?
  0.736 | Bagaimana cara melacak paket yang sedang dikirim?

JAWABAN AI:
Halo! Untuk mengetahui posisi paket Anda, Anda bisa melacaknya menggunakan nomor resi yang tersedia di halaman 'Pesanan Saya' di website kurir terkait.

Jika paket Anda belum sampai melebihi estimasi pengiriman lebih dari 3 hari, mohon hubungi customer service kami dengan menyertakan nomor resi agar dapat kami bantu telusuri lebih lanjut.
------------------------------------------------------------


In [17]:
test_queries = [
    ("Jelas ada #1", "Bagaimana cara membatalkan pesanan yang sudah dibuat?"),
    ("Jelas ada #2", "Apakah saya bisa mengubah alamat pengiriman sesuai dengan yang saya inginkan?"),
    ("Parafrase #1", "Paket saya dimana, ko nda nyampe?"),
    ("Parafrase #2", "Kalau barangnya rusak, uangnya bisa balik nggak?"),
    ("Tidak ada di KB", "Apakah kalian menjual tiket konser?"),
]

log = []
for tipe, q in test_queries:
    print("###", tipe)
    results, jawaban = answer_query(q)
    log.append({
        "tipe": tipe,
        "query": q,
        "top1_faq": results[0]["question"],
        "top1_score": round(results[0]["score"], 3),
        "jawaban": jawaban,
    })

pd.DataFrame(log)[["tipe", "query", "top1_faq", "top1_score"]]

### Jelas ada #1
QUERY: Bagaimana cara membatalkan pesanan yang sudah dibuat?
FAQ yang ditemukan:
  0.782 | Bagaimana cara membatalkan pesanan yang sudah dibuat?
  0.705 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
  0.692 | Kenapa pesanan saya otomatis dibatalkan sistem?

JAWABAN AI:
Halo! Tentu, saya bantu jelaskan caranya.

Anda bisa membatalkan pesanan melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Namun, perlu diingat bahwa setelah status pesanan berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan.
------------------------------------------------------------
### Jelas ada #2
QUERY: Apakah saya bisa mengubah alamat pengiriman sesuai dengan yang saya inginkan?
FAQ yang ditemukan:
  0.791 | Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
  0.707 | Apakah bisa memilih jasa pengiriman sendiri saat checkout?
  0.677 | Apakah bisa menggabungkan beberapa pesanan menjadi satu pengiriman?

JAWABAN AI:
Tentu, Anda bisa mengub

,tipe,query,top1_faq,top1_score
0,Jelas ada #1,Bagaimana cara membatalkan pesanan yang sudah ...,Bagaimana cara membatalkan pesanan yang sudah ...,0.782
1,Jelas ada #2,Apakah saya bisa mengubah alamat pengiriman se...,Apakah saya bisa mengubah alamat pengiriman se...,0.791
2,Parafrase #1,"Paket saya dimana, ko nda nyampe?",Bagaimana jika paket saya belum sampai melebih...,0.758
3,Parafrase #2,"Kalau barangnya rusak, uangnya bisa balik nggak?",Bagaimana prosedur pengembalian barang yang ti...,0.738
4,Tidak ada di KB,Apakah kalian menjual tiket konser?,Apakah bisa memesan dalam jumlah besar untuk k...,0.620


In [18]:
with open("../documentation/retrieval_test_results.md", "w", encoding="utf-8") as f:
    f.write("# Hasil Pengujian Retrieval dan Response\n\n")
    for item in log:
        f.write(f"## {item['tipe']}\n")
        f.write(f"- **Query:** {item['query']}\n")
        f.write(f"- **FAQ top-1:** {item['top1_faq']} (skor {item['top1_score']})\n\n")
        f.write(f"**Jawaban AI:**\n\n{item['jawaban']}\n\n---\n\n")
print("Tersimpan")

Tersimpan


In [19]:
cek = [
    "membatalkan pesanan yang sudah dibuat",
    "mengubah alamat pengiriman setelah",
    "melacak paket yang sedang dikirim",
    "pengembalian barang yang tidak sesuai",
]
for kata in cek:
    for _, r in df[df["question"].str.contains(kata)].iterrows():
        print("Q:", r["question"])
        print("A:", r["answer"])
        print()

Q: Bagaimana cara membatalkan pesanan yang sudah dibuat?
A: Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Setelah status berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan.

Q: Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
A: Perubahan alamat hanya bisa dilakukan sebelum pesanan berstatus 'Dikirim'. Silakan hubungi customer service secepatnya jika ingin mengubah alamat.

Q: Bagaimana cara melacak paket yang sedang dikirim?
A: Nomor resi akan muncul di halaman 'Pesanan Saya' setelah barang dikirim, dan dapat digunakan untuk melacak posisi paket di website kurir terkait.

Q: Bagaimana prosedur pengembalian barang yang tidak sesuai?
A: Ajukan pengembalian melalui menu 'Pesanan Saya' dengan memilih alasan dan menyertakan bukti foto, maksimal 3 hari sejak barang diterima.



In [20]:
# 1. Apakah ada FAQ soal refund/pengembalian dana di knowledge base?
mask = df["question"].str.contains("refund|dana|uang", case=False) | \
       df["answer"].str.contains("refund|dana|uang", case=False)
for _, r in df[mask].iterrows():
    print(f"[{r['category']}] Q: {r['question']}")
    print("A:", r["answer"])
    print()

print("=" * 60)

# 2. Dari mana frasa "cacat produksi" berasal?
mask2 = df["answer"].str.contains("cacat", case=False)
for _, r in df[mask2].iterrows():
    print(f"Q: {r['question']}")
    print("A:", r["answer"])
    print()

[Pengembalian] Q: Berapa lama proses refund setelah pengembalian disetujui?
A: Dana refund biasanya diproses dalam 3-7 hari kerja setelah barang dikembalikan diterima dan diverifikasi penjual.

[Pengembalian] Q: Ke mana dana refund akan dikembalikan?
A: Dana refund akan dikembalikan ke metode pembayaran awal, atau ke saldo akun apabila memilih opsi tersebut saat pengajuan.

[Pengembalian] Q: Apakah bisa menukar barang dengan ukuran/warna lain daripada refund?
A: Penukaran ukuran/warna tergantung kebijakan masing-masing penjual, silakan cek ketersediaan stok terlebih dahulu sebelum mengajukan.

[Produk] Q: Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
A: Jika stok habis setelah pesanan dibuat, penjual akan membatalkan sebagian/seluruh pesanan dan dana otomatis dikembalikan.

Q: Apakah semua produk bisa dikembalikan?
A: Produk seperti pakaian dalam, produk higienis, dan barang custom umumnya tidak dapat dikembalikan kecuali terbukti cacat produksi.

Q: Apakah biaya ongko

In [21]:
q = "Kalau barangnya rusak, uangnya bisa balik nggak?"
for n, r in enumerate(semantic_search(q, k=8), start=1):
    print(f"{n}. {r['score']:.3f} | [{r['category']}] {r['question']}")

print()
_ = answer_query(q, k=5)

1. 0.738 | [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
2. 0.732 | [Pengembalian] Apakah biaya ongkos kirim pengembalian ditanggung pembeli?
3. 0.723 | [Produk] Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
4. 0.723 | [Pengembalian] Apakah semua produk bisa dikembalikan?
5. 0.716 | [Pengembalian] Saya menerima barang yang salah, apa yang harus saya lakukan?
6. 0.715 | [Pengembalian] Apakah bisa menukar barang dengan ukuran/warna lain daripada refund?
7. 0.702 | [Pengiriman] Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?
8. 0.700 | [Pengembalian] Berapa lama proses refund setelah pengembalian disetujui?

QUERY: Kalau barangnya rusak, uangnya bisa balik nggak?
FAQ yang ditemukan:
  0.738 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
  0.732 | Apakah biaya ongkos kirim pengembalian ditanggung pembeli?
  0.723 | Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
  0.723 | Apakah semua produk bisa dike